Exercise: How does the accuracy of the 3 nearest neighbour classifier change with the number of splits? How is it affected by the split size? Compare the results with the 1 nearest neighbour classifier.

More splits do not inherently make KNN more accurate; they make the evaluation more reliable because every sample is tested across folds. Compare 1-NN and 3-NN with stratified cross-validation:

In [4]:
from sklearn.datasets import load_iris
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier

iris = load_iris()
X, y = iris.data, iris.target

for n_splits in [3, 5, 10]:
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    for k in [1, 3]:
        model = KNeighborsClassifier(n_neighbors=k)
        scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")

        print(
            f"{n_splits}-fold CV, {k}-NN: "
            f"{scores.mean():.2%} ± {scores.std():.2%}"
        )

3-fold CV, 1-NN: 95.33% ± 4.11%
3-fold CV, 3-NN: 96.00% ± 3.27%
5-fold CV, 1-NN: 95.33% ± 4.99%
5-fold CV, 3-NN: 95.33% ± 4.99%
10-fold CV, 1-NN: 96.00% ± 5.33%
10-fold CV, 3-NN: 96.00% ± 5.33%


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

for test_size in [0.2, 0.3, 0.4, 0.5]:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y
    )

    for k in [1, 3]:
        model = KNeighborsClassifier(n_neighbors=k)
        model.fit(X_train, y_train)
        accuracy = accuracy_score(y_test, model.predict(X_test))

        print(f"Test size {test_size:.0%}, {k}-NN: {accuracy:.2%}")

Test size 20%, 1-NN: 96.67%
Test size 20%, 3-NN: 100.00%
Test size 30%, 1-NN: 93.33%
Test size 30%, 3-NN: 95.56%
Test size 40%, 1-NN: 95.00%
Test size 40%, 3-NN: 96.67%
Test size 50%, 1-NN: 96.00%
Test size 50%, 3-NN: 92.00%


Expected conclusion: 3-NN is usually a little more stable than 1-NN because voting among three neighbors reduces sensitivity to an unusual training sample. Larger test splits leave less training data, which can slightly reduce accuracy; smaller test splits give less reliable accuracy estimates. On Iris, both models normally achieve very high accuracy, so differences may be small.


Question 1:

Does averaging the validation accuracy across multiple splits give more consistent results?


Averaging validation accuracy across multiple splits gives a more consistent, trustworthy estimate of model performance.
A single train/test split can be unusually easy or difficult depending on which samples land in the test set. Using multiple splits—such as 5-fold or 10-fold cross-validation—reduces that randomness by evaluating the classifier on several different validation sets.


In [6]:
mean_accuracy = scores.mean()
variation = scores.std()

print(f"Accuracy: {mean_accuracy:.2%} ± {variation:.2%}")

Accuracy: 96.00% ± 5.33%


 Question 2:

 Does it give more accurate estimate of test accuracy?


 Averaging validation accuracy over multiple splits gives a more accurate estimate of how the model will perform on unseen data than relying on one train/test split. It reduces the chance that an unusually easy or difficult test set distorts the result.
Cross-validation is especially helpful with small datasets like Iris. Still, keep a final untouched test set if you need one unbiased performance estimate after choosing model settings.

Question 3:

What is the effect of the number of iterations on the estimate? Do we get a better estimate with higher iterations?

More iterations generally give a more stable estimate of accuracy because the result is averaged over more random train/test splits.
With only a few iterations, the estimate can vary noticeably depending on which samples happen to be in the test set. As iterations increase, the mean accuracy tends to settle toward the model’s typical performance and the uncertainty decreases.
However, improvements diminish after a point: moving from 1 to 10 iterations helps much more than moving from 100 to 1,000. More iterations cost more computation but do not improve the classifier itself—only confidence in the performance estimate.


Question 4:

Consider the results you got for the previous questions. Can we deal with a very small train dataset or validation dataset by increasing the iterations?

Not fully. Increasing iterations reduces randomness in the estimated accuracy, but it cannot replace missing data.
- A very small training set produces a weaker model because it has too few examples to learn from.
- A very small validation set gives a noisy accuracy measurement because each individual prediction has too much influence.
More iterations average out some split-related variation, so the estimate becomes steadier. But the model still trains on small samples, and each validation result still contains limited information.
A better approach is cross-validation—such as stratified 5-fold or 10-fold cross-validation—which uses the available data efficiently while keeping class proportions balanced.

 Exercise: Try to implement a 3 nearest neighbour classifier and compare the accuracy of the 1 nearest neighbour classifier and the 3 nearest neighbour classifier on the test dataset. You can use the KNeighborsClassifier class from the scikit-learn library to implement the K-Nearest Neighbors model. You can set the number of neighbors using the n_neighbors parameter. You can also use the accuracy_score function from the scikit-learn library to calculate the accuracy of the model.

In [7]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Load data and create the same train/test split for both models
iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 1-nearest-neighbor classifier
knn_1 = KNeighborsClassifier(n_neighbors=1)
knn_1.fit(X_train, y_train)
accuracy_1 = accuracy_score(y_test, knn_1.predict(X_test))

# 3-nearest-neighbor classifier
knn_3 = KNeighborsClassifier(n_neighbors=3)
knn_3.fit(X_train, y_train)
accuracy_3 = accuracy_score(y_test, knn_3.predict(X_test))

print(f"1-NN accuracy: {accuracy_1:.2%}")
print(f"3-NN accuracy: {accuracy_3:.2%}")

1-NN accuracy: 93.33%
3-NN accuracy: 95.56%
